# VELALAR COLLEGE OF ENGINEERING AND TECHNOLOGY (AUTONOMOUS)
## DEPARTMENT OF INFORMATION TECHNOLOGY
### ACADEMIC YEAR: 2026 - 2027 (ODD) | CLASS: 25IT3A
**Course Activity:** Project-Based Assessment  
**Scenario 9:** Retail Store Inventory and Sales Analysis System  
**Student Name:** Ashvika K  
**Register Number:** 732925ITR009  

---
### Objective (Requirement 6 & 10):
Retrieve the project dataset from the relational database using Python, preprocess the data, and generate meaningful visualizations to represent sales patterns, stock levels, and revenue trends.

## Step 1: Import Required Libraries and Connect to Database

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options and plot styling
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid")

# Connect to SQLite Database
db_path = '../database/retail_store.db'
conn = sqlite3.connect(db_path)
print("Connected successfully to:", db_path)

## Step 2: Retrieve Datasets from Relational Database (SQL Extraction)

In [ ]:
# Extract Products with Category Information
query_products = """
SELECT 
    p.product_id, p.product_name, c.category_name, p.brand,
    p.cost_price, p.selling_price, p.stock_quantity, p.reorder_level, p.unit_of_measure
FROM products p
JOIN categories c ON p.category_id = c.category_id
"""
df_products = pd.read_sql_query(query_products, conn)

# Extract Transactions and Items
query_transactions = """
SELECT 
    t.transaction_id, t.transaction_date, t.subtotal, t.tax_amount,
    t.discount_amount, t.grand_total, t.payment_method, t.status,
    ti.product_id, ti.quantity, ti.unit_price, ti.item_subtotal
FROM transactions t
JOIN transaction_items ti ON t.transaction_id = ti.transaction_id
WHERE t.status = 'COMPLETED'
"""
df_sales = pd.read_sql_query(query_transactions, conn)

print(f"Retrieved {len(df_products)} products and {len(df_sales)} sales line items.")
df_products.head()

## Step 3: Data Preprocessing and Feature Engineering (Pandas)

In [ ]:
# 1. Convert Dates
df_sales['transaction_date'] = pd.to_datetime(df_sales['transaction_date'])
df_sales['sale_date'] = df_sales['transaction_date'].dt.date

# 2. Calculate Profit Margin per product
df_products['unit_margin'] = df_products['selling_price'] - df_products['cost_price']
df_products['margin_percentage'] = (df_products['unit_margin'] / df_products['selling_price']) * 100
df_products['stock_valuation'] = df_products['stock_quantity'] * df_products['selling_price']

# 3. Stock Status Categorization
def classify_stock(row):
    if row['stock_quantity'] == 0:
        return 'OUT_OF_STOCK'
    elif row['stock_quantity'] <= row['reorder_level']:
        return 'LOW_STOCK'
    else:
        return 'IN_STOCK'

df_products['stock_status'] = df_products.apply(classify_stock, axis=1)
df_products[['product_id', 'product_name', 'stock_quantity', 'reorder_level', 'stock_status']].head(10)

## Step 4: Inventory Analysis - Low Stock Alert Report (FR5)

In [ ]:
low_stock_df = df_products[df_products['stock_quantity'] <= df_products['reorder_level']]
print("CRITICAL REORDER ALERTS:")
low_stock_df[['product_id', 'product_name', 'category_name', 'stock_quantity', 'reorder_level', 'stock_status']]

## Step 5: Visualizations & Trend Analysis (Requirement 10)

In [ ]:
# Visualization 1: Daily Revenue Trend
daily_rev = df_sales.groupby('sale_date')['grand_total'].first().reset_index() # Unique txn totals

plt.figure(figsize=(10, 4.5))
plt.plot(daily_rev['sale_date'].astype(str), daily_rev['grand_total'], marker='o', color='#2563eb', linewidth=2.5)
plt.title('Daily Sales Revenue Trend (Nov 2026)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Revenue (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Top Selling Products by Volume
top_products = df_sales.merge(df_products, on='product_id').groupby('product_name')['quantity'].sum().nlargest(5)

plt.figure(figsize=(9, 4.5))
sns.barplot(x=top_products.values, y=top_products.index, palette='crest')
plt.title('Top 5 Selling Products by Volume', fontsize=14, fontweight='bold')
plt.xlabel('Units Sold')
plt.ylabel('Product Name')
plt.tight_layout()
plt.show()

In [ ]:
# Close Database Connection
conn.close()
print("Analysis completed. Database connection closed safely.")